In [0]:
%pip install pycoingecko --quiet

from pycoingecko import CoinGeckoAPI
from pyspark.sql import functions as F
from datetime import datetime, timezone
import pandas as pd



cg = CoinGeckoAPI()   

RUN_TS = datetime.now(timezone.utc)
RUN_DT = RUN_TS.strftime("%Y-%m-%d")
RUN_ID = RUN_TS.strftime("%Y%m%d_%H%M%S")
TABLE  = "crypto_db.bronze_trending_data"
GOLD   = "crypto_db.gold_trending_coins"

# ── Fetch /search/trending ─────────────────────────────────────────
data = cg.get_search_trending()

# ── Extract — iterate through JSON array as required ──────────────
records = []
for item in data.get("coins", []):          # top 7 trending coins
    coin = item.get("item", {})
    records.append({
        "coin_id":         coin.get("id"),
        "name":            coin.get("name"),
        "symbol":          coin.get("symbol"),
        "market_cap_rank": coin.get("market_cap_rank"),
        "thumb":           coin.get("thumb"),
        "score":           coin.get("score"),    # 0 = #1 trending
        "trending_rank":   coin.get("score", 0) + 1,
        "ingestion_date":  RUN_DT,
        "ingestion_ts":    RUN_TS.strftime("%Y-%m-%d %H:%M:%S"),
        "run_id":          RUN_ID,
    })

print(f"Trending coins fetched: {len(records)}")
for r in records:
    print(f"  #{r['trending_rank']}  {r['name']} ({r['symbol']})  MCap Rank: {r['market_cap_rank']}")

# ── Build DataFrame ────────────────────────────────────────────────
pdf = pd.DataFrame(records)
sdf = spark.createDataFrame(pdf)

# ── Idempotent MERGE into Bronze ──────────────────────────────────
sdf.createOrReplaceTempView("_stg_trending")
if not spark.catalog.tableExists(TABLE):
    (sdf.write.format("delta").mode("overwrite")
        .saveAsTable(TABLE))
    print(f"Created {TABLE}")
else:
    spark.sql(f"""
        MERGE INTO {TABLE} tgt USING _stg_trending src
        ON tgt.coin_id = src.coin_id
        AND tgt.ingestion_date = src.ingestion_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into {TABLE}")

# ── Enrich and write Gold table ───────────────────────────────────
snap = spark.table("crypto_db.gold_market_snapshot")

df_gold = (sdf
    .join(snap.select("id","current_price","market_cap",
                      "total_volume","price_change_percentage_24h",
                      "performance_label","trading_signal"),
          sdf["coin_id"] == snap["id"], "left")
    .withColumn("trending_rank",    F.col("trending_rank").cast("integer"))
    .withColumn("market_cap_rank",  F.col("market_cap_rank").cast("integer"))
    .withColumn("gold_updated_at",  F.lit(RUN_TS.strftime("%Y-%m-%d %H:%M:%S")).cast("timestamp"))
    .select(
        "trending_rank","coin_id","name","symbol",
        "market_cap_rank",
        "current_price","market_cap","total_volume",
        "price_change_percentage_24h",
        "performance_label","trading_signal",
        "ingestion_date","gold_updated_at"
    )
)

if not spark.catalog.tableExists(GOLD):
    df_gold.write.format("delta").mode("overwrite").saveAsTable(GOLD)
    print(f"Created {GOLD}")
else:
    df_gold.createOrReplaceTempView("_stg_gold_trending")
    spark.sql(f"""
        MERGE INTO {GOLD} tgt USING _stg_gold_trending src
        ON tgt.coin_id = src.coin_id
        AND tgt.ingestion_date = src.ingestion_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into {GOLD}")

spark.table(GOLD).show(7, truncate=False)